# 학습 후보 8장 · 수행자 박스와 가까운 쪽 관절 판독
a01·b01·b04·c02의 이륙 후보와 무릎 통과 프레임을 한 장씩 엽니다. 여덟 장 모두 선수 본인의 **왼쪽**이 카메라에 가깝습니다. 프레임은 0부터 셉니다.

**모든 셀 실행(Run All Cells)** 후 2번 화면을 사용하세요. b04의 견봉은 지난 직접 판독, 골반은 소켓 1·2차 추정 좌표의 중점으로 시작합니다. 재실행하면 이번에 저장한 수정값이 우선합니다.

### 1. 기존 판독과 진행 상황 불러오기

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from wlcv.annotation.performer_box import PerformerBox
from wlcv.annotation.pose_points import pose_point_editor
from wlcv.annotation.train_pose import candidate_frames, import_b04, progress_table
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS

frames = candidate_frames(ROOT / "data/metadata", SPLIT)
folder = ANNOTATIONS / "train_pose_candidates"
import_b04(ANNOTATIONS, folder)
progress = progress_table(frames, folder)
display(progress)

### 2. 프레임을 골라 박스와 관절 표시하기
1. **수행자 박스**에서 머리부터 양발까지 보이는 몸 전체를 감싸도록 모서리 한 곳과 대각선 반대편을 차례로 클릭하세요. 두 번째 클릭에서 초록 박스가 저장됩니다. 바꾸려면 두 모서리를 다시 찍으세요.
2. 선수의 왼쪽 **어깨(견봉) · 골반(소켓) · 무릎(관절선) · 발목(복사뼈)** 중 하나를 고르세요. 보이는 점은 **직접 확인**, 가림이나 소켓 위치를 짐작한 점은 해당 **추정** 사유를 고른 뒤 클릭하세요. 같은 관절을 다시 찍으면 수정됩니다.
3. 점을 짚을 수 없으면 **가려서 판독 불가** 또는 **위치 불확실**을 누르세요. 좌표는 비워 두고 판독 사유를 저장합니다. 십자는 직접 확인, 마름모는 추정입니다. 작은 관절은 **원본 크기로 확대**해서 찍을 수 있습니다.

초록 박스와 네 관절의 판독 상태를 모두 기록하면 한 장이 완료됩니다. 중단해도 다시 열면 첫 미완료 장면에서 이어집니다. 장면 목록에서 완료한 프레임도 열어 볼 수 있습니다.

In [ ]:
pending = progress.index[~progress["완료"]].tolist()
selector = widgets.Dropdown(
    description="장면",
    options=[(f"{r.person_id} · {r.frame}번 · {r.phase}", r.Index) for r in frames.itertuples()],
    value=pending[0] if pending else 0,
)
panel = widgets.VBox()

def open_frame(change=None):
    panel.children = ()
    row = frames.loc[selector.value]
    box = PerformerBox(folder, row.video_id, row.frame)
    editor = pose_point_editor(
        VIDEOS / row.filename, row.frame, row.person_id, row.side, split="train",
        annotations=folder, trunk_review=True, performer_box=box,
        frame_label=f"{row.video_id} · {row.frame}번 · {row.phase}",
    )
    panel.children = (editor,)

selector.observe(open_frame, names="value")
open_frame()
display(selector, panel)

### 3. 저장본 확인
판독 후 이 셀을 다시 실행하면 장면별 저장 상태와 완료 장수가 갱신됩니다. 박스는 `data/annotations/train_pose_candidates/person_boxes/`에 원본 픽셀 기준 두 모서리로, 관절은 같은 상위 폴더의 `personal_pose_points/`에 좌표·좌우·상태·사유로 저장됩니다. **미검수**는 아직 아무 결정도 저장하지 않은 항목입니다.

In [ ]:
saved = progress_table(frames, folder)
print(f"{saved['완료'].sum()} / {len(saved)}장 완료")
display(saved)